# Kimonu on the networks of Figures 1 and 2

Kimonu finds concordant complexes, kinetic modules, absolute concentration robustness (ACR) and absolute
concentration ratio robustness (ACRR) from exact nullspaces, and gives the values these relations take at given
rate constants. This notebook runs every call on the two small networks of the paper. Everything is exact rational
arithmetic and takes a few seconds.

In [ ]:
from fractions import Fraction
from math import prod
import numpy as np
import kimonu


def network(reactions):
    """The kimonu.CRN and the complex names of reactions written as 'C + 2P -> 3P' ('0' is the zero complex).

    cobra adds up the two sides of a reaction (C + 2P -> 3P becomes C -> P), which loses the complexes that
    mass action needs, so small networks are written down directly here; for genome-scale models,
    kimonu.prepare(model) does the same from a cobra model."""
    def side(s):
        comp = {}
        for term in s.split("+"):
            term = term.strip()
            if term != "0":
                n = len(term) - len(term.lstrip("0123456789"))
                comp[term[n:]] = comp.get(term[n:], 0) + (int(term[:n]) if n else 1)
        return comp
    sides = [tuple(side(s) for s in r.split("->")) for r in reactions]
    species = list(dict.fromkeys(sp for s, p in sides for sp in [*s, *p]))
    complexes = list(dict.fromkeys(tuple(sorted(c.items())) for pair in sides for c in pair))
    A = np.zeros((len(complexes), len(sides)), dtype=int)
    Y = np.zeros((len(species), len(complexes)), dtype=int)
    for j, (s, p) in enumerate(sides):
        A[complexes.index(tuple(sorted(s.items()))), j] -= 1
        A[complexes.index(tuple(sorted(p.items()))), j] += 1
    for c, comp in enumerate(complexes):
        for sp, n in comp:
            Y[species.index(sp), c] = n
    names = [" + ".join((f"{n}{sp}" if n > 1 else sp) for sp, n in c) or "0" for c in complexes]
    reactions = [str(j + 1) for j in range(len(sides))]          # R1, R2, ... as in the figure legends
    return kimonu.crn_from_matrices(A, Y, metabolite_ids=species, reaction_ids=reactions), names

## Figure 1

Fourteen reactions among nine species, no exchange with the outside.

In [ ]:
fig1 = ["A -> B", "B -> A", "B -> C", "C -> B", "C -> D", "D + E -> F", "F -> D + E", "F -> B + G",
        "C + G -> H", "H -> C + G", "H -> C + E", "A + G -> I", "I -> A + G", "I -> A + E"]
crn, cx = network(fig1)
named = lambda groups: [sorted(cx[c] for c in g) for g in groups if len(g) > 1]

conc = kimonu.concordance(crn)
print("concordant complexes:", named(conc.modules))
print("balanced complexes:  ", sorted(cx[c] for c in conc.balanced))

**Kinetic modules, ACR and ACRR.** The iterated route, Kimonu's default, proves relations that hold at every
positive steady state, but it can miss some. The tangent route linearizes mass action at one steady state and
misses none, but it can report false couplings. `certify` runs both at one reference flux. Here they agree, so the
list is exact.

In [ ]:
it = kimonu.iterated_route(crn)
print("kinetic modules:", named(it.modules))
print("ACR:", it.acr, "  ACRR pairs:", it.acrr)
print("routes agree, so the list is complete:", kimonu.certify(crn).certified)

**Values at given rate constants.** `relation_values` reruns the iterated route at k and returns each value
exactly. G is fixed by a rational function of the rate constants that flux coupling alone does not give, so
`binomial_formulas` has no monomial for it.

In [ ]:
k = list(range(1, 15))
print("G at k = 1..14:", kimonu.relation_values(crn, k).acr)
print("monomial from flux coupling:", kimonu.binomial_formulas(crn).acr["G"])

**Rate constants for chosen values.** The routes ran at an interior flux r* (`it.flux`); at k = r* the
steady state is x = 1. `rate_constants` moves it to any x > 0, so every relation takes the value x gives it.

In [ ]:
x = [Fraction(1)] * len(crn.metabolite_ids)
x[crn.metabolite_ids.index("G")] = Fraction(3)
k = kimonu.rate_constants(crn, it.flux, x)
print("G at k = r*/psi(x):", kimonu.relation_values(crn, k).acr)

## Figure 2

Protein X is made and degraded, dimerizes into an inactive kinase C, and active P activates C cooperatively.
The reactions are numbered as in the legend of Figure 2. R1 supplies X, R2 degrades X, R3 is the dimerization
2X → C, R4 is C + 2P → 3P, and R5 and R6 are the decay of C and of P. At these rate constants there are two
steady states, (X, C, P) = (10, 1, 4) and (10, 4, 1).

In [ ]:
fig2 = ["0 -> X", "X -> 0", "2X -> C", "C + 2P -> 3P", "C -> 0", "P -> 0"]       # R1 to R6
K = [20, 1, Fraction(1, 20), Fraction(1, 4), 1, 1]
crn, cx = network(fig2)
named = lambda groups: [sorted(cx[c] for c in g) for g in groups if len(g) > 1]

species, N, substrate = crn.metabolite_ids, crn.N.to_numpy(), kimonu.io.substrate_complexes(crn)
for state in ({"X": 10, "C": 1, "P": 4}, {"X": 10, "C": 4, "P": 1}):
    x = [Fraction(state[s]) for s in species]
    rates = [K[j] * prod(x[i] ** int(y) for i, y in crn.Yt[substrate[j]]) for j in range(len(fig2))]
    print(state, "dx/dt =", [str(sum(int(N[i, j]) * rates[j] for j in range(len(fig2)))) for i in range(len(species))])

**The two routes part.** The iterated route proves only the module {P, C + 2P}. The tangent route sees no
direction to move in at either steady state and calls X, C and P fixed. But C and P differ between the two
steady states, so their ACR is false. X is truly fixed, and the iterated route misses it.
`certify` reports the gap as a warning and lists the relations that stay open.

In [ ]:
cert = kimonu.certify(crn, detach_uptakes=False)
print("iterated: modules", named(cert.iterated.modules), " ACR", cert.iterated.acr)
print("tangent:  modules", named(cert.tangent.modules), " ACR", cert.tangent.acr,
      " dim ker(W Y^T) =", cert.tangent.dimension)
print("open ACR:", cert.open_acr)

**Values.** In the module {C + 2P, P}, ψ(P) / ψ(C + 2P) = 1 / (C·P) = 1/4 at both steady states. It comes from
flux coupling. R4 and R6 carry the same flux, so k4 C P² = k6 P and 1 / (C·P) = k4 / k6.

In [ ]:
v = kimonu.relation_values(crn, K, detach_uptakes=False)
b = kimonu.binomial_formulas(crn, detach_uptakes=False)
for (ref, ratios), (_, monomials) in zip(v.modules, b.modules):
    for c in ratios:
        print(f"psi({cx[c]}) / psi({cx[ref]}) = {ratios[c]} = {monomials[c].format(crn.reaction_ids)}")

**Why X is missed.** The balance of X alone, k1 - k2 X - 2 k3 X² = 0, fixes X = (√(k2² + 8 k1 k3) - k2) / 4 k3
= 10 at both steady states. This is the root of a quadratic. Every value the iterated route can derive is
a product of powers of rational functions of k, and this one is not.

In [ ]:
from math import isqrt
k1, k2, k3 = K[:3]
disc = k2 ** 2 + 8 * k1 * k3
root = isqrt(int(disc))
assert root * root == disc
print("X =", (root - k2) / (4 * k3), "  (k2^2 + 8 k1 k3 =", disc, ")")